OPZETTEN TABEL


In [13]:
#imports
debugging_mode=True
from pyspark.sql.functions import *
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql.types import LongType, IntegerType, ByteType, StringType
from datetime import datetime
import ConnectionConfig as cc

In [14]:
#config
cc.setupEnvironment()
spark = cc.startLocalCluster("DIM_TREASURE_TYPE",4)
spark.getActiveSession()

Environment variables are set...


In [15]:
#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")
cc.config.sections()

['default', 'tutorial_op', 'catchem', 'kafka']

In [16]:
#EXTRACT

#get info
#treasure
df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("treasure")

#stage
df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_stages") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("stages")

In [17]:
#TRANSFORM

#select maken voor info voor de tabel
df_dim_treasure_type = spark.sql("""
    WITH stages_count AS (
        SELECT
            treasure_id,
            COUNT(DISTINCT stages_id) AS size
        FROM stages
        GROUP BY treasure_id
    ),
    treasure_with_size AS (
        SELECT
            t.difficulty,
            t.terrain,
            COALESCE(s.size, 0) AS size
        FROM treasure t
        LEFT JOIN stages_count s ON t.id = s.treasure_id
    )
    SELECT DISTINCT
        difficulty AS Difficulty,
        CASE difficulty
            WHEN 0 THEN 'Very Easy'
            WHEN 1 THEN 'Easy'
            WHEN 2 THEN 'Medium'
            WHEN 3 THEN 'Hard'
            WHEN 4 THEN 'Super Hard'
        END AS DifficultyName,
        terrain AS Terrain,
        size AS Size
    FROM treasure_with_size
    ORDER BY difficulty, terrain, size
""")


In [18]:
#LOAD

#deltatabel maken
spark.sql("DROP TABLE IF EXISTS dimTreasureType")

DeltaTable.createOrReplace(spark) \
    .tableName("dimTreasureType") \
    .addColumn("TreasureTypeSurKey", LongType(), nullable=False, generatedAlwaysAs=IdentityGenerator(0, 1)) \
    .addColumn("Difficulty", IntegerType()) \
    .addColumn("DifficultyName", StringType()) \
    .addColumn("Terrain", IntegerType()) \
    .addColumn("Size", LongType()) \
    .property("delta.feature.identityColumns", "supported") \
    .execute()

In [ ]:
#LOAD

df_dim_treasure_type.write.format("delta").mode("append").saveAsTable("dimTreasureType")

spark.sql("SELECT * FROM dimTreasureType").show(100)

In [ ]:
spark.stop()